# Assignment 3
## Clustering Models
## CSCI E-108    


**Instructions:** Complete the exercises in this notebook. Some exercises require coding. The answers to the exercise questions should be short and succinct. **Most question answers require only one to three sentences**. Use of overly long answers will result in loss of credit. Please thin about the theory behind the question and apply the reasoning required to create an answer. Do not use an LLM or other AI to create your answers. 

> **Note on packages used:** The code in this notebook uses two widely-used open-source data science Python packages, Scikit-Learn and Seaborn. Some links to quick-start guides and tutorials are provided here. Please ask you instructional staff if you have additional questions or difficulties using these packages.     
> 1. The [Scikit-Learn package](https://scikit-learn.org/stable/index.html) is a core machine learning package. You can find a quick-start guide for Scikit-Learn [here](https://scikit-learn.org/stable/getting_started.html). An overview of [applying Scikit-Learn for clustering models](https://scikit-learn.org/stable/modules/clustering.html) is provided under the unsupervised learning documentation.
> 2. The [HDBSCAN pacage](https://hdbscan.readthedocs.io/en/latest/index.html) contains code for efficient large-scale construction and evaluation of sophisticated state-of-the-art density clustering models.   
> 3. The [Seaborn package](https://seaborn.pydata.org/index.html#) is a powerful data visualization package. You can find a quick-start guide to Seaborn [here](https://seaborn.pydata.org/introduction.html). Seaborn provides uses with many examples and [tutorials](https://seaborn.pydata.org/tutorial.html) on each category of charts provided.  
> 4. The Uniform Manifold Approximation and Projection or [UMAP package](https://umap-learn.readthedocs.io/en/latest/) allows  visualization of high-dimensional data on a 2-dimensional **manifold**. You can find installation instructions for this package [here](https://umap-learn.readthedocs.io/en/latest/index.html). We will discuss manifold learning in the dimensionality reduction lesson. For the current version of the umap package it is recommend that you also install the umap-learn package by un-commenting and running the code below.
>
> You may need to use the --update option if you have older versions of these packages installed.  

In [ ]:
#!pip install seaborn
#!pip install hdbscan
#!pip install umap
#!pip install umap-learn
#!pip install seaborn 
#!pip install networkx

## Introduction    

**Clustering models** are core data mining methods. Clustering models are also known as **unsupervised learning** models. The goal of these models is to extract structure and relationships from complex data. Carrying out this type of exploration is difficult since there is no ground truth as a basis of comparison. 

In this assignment you will work with a complex human resources data set to explore the basic concepts of clustering. Specifically, you will use the [human resources (HR) dataset from Kaggle](https://www.kaggle.com/jacksonchou/hr-analytics). The business goal is to discover attributes common to employees who leave a large company prematurely. You will use multiple clustering algorithms to build an understanding of the relationships between the variables. Since we do not know in advance which attributes are important in an employees decision to leave, this is a perfect application for clustering methods.   

Evaluation of any unsupervised learning model is a difficult task at the best of times. In this assignment you will use a combination of numeric and graphical evaluation methods. The scope of model evaluation and comparison has deliberately been limited here. Many more methods can and should be applied to complex real-world problems. Regardless of the evaluation methods applied, interpretation of clustering model results remains difficult and requires judgment. 

As you proceed with the exercises, keep in mind that the relationships in this data set are complex. There are unlikely to be one or just a few simple reasons why long-term employees chooses to leave their jobs. Consequently, you should not expect simple well defined results for any analytical model. This type of difficulty and complexity is inherent in many real-world applications of unsupervised learning.    

The cluster models you will apply in this assignment have many hyperparameters. To limit the scope of the exercises, values for many hyperparameters are specified in many cases. These values were found by limited exploration of the hyperparameter space. For a real-world project a great deal more time-consuming exploration would be required. 

> **Important Note:** You will need at least 2 GB of free RAM to execute the exercises in this notebook. 


### Load and explore the data   

As a first step in this analysis, execute the code in the cell below to import the required packages. 

> **Note:** The code in the cell below contains two blocks that provide patches to deal with version incompatabilities between Python packages Scikit-Learn, UMAP and HDBSCAN. Some environments and package version combinations may not need these patches. If you encounter exceptions arising from version incompatibility you can comment out one or both of these blocks, restart the notebook, and test the code again  

In [ ]:
import pandas as pd
import numpy as np
import numpy.random as nr
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering, DBSCAN, OPTICS, MiniBatchKMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.pairwise import pairwise_distances, cosine_similarity, cosine_distances, euclidean_distances 
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.datasets import load_iris
from sklearn.neighbors import radius_neighbors_graph, kneighbors_graph
from sklearn.decomposition import PCA
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import time
import math

import sklearn.utils
import sklearn.utils.validation

import hdbscan

import sys

%matplotlib inline

In [ ]:
import sklearn
print(sklearn.__version__)
print(umap.__version__)

Next, execute the code in the cell below to import the dataset and display the data types of the columns.  

In [ ]:
HR_data = pd.read_csv('../data/HR_comma_sep.csv')
HR_data.dtypes

There are several data types here. The first two variables have floating point numeric values. There are several integer numeric values. And, finally the last two columns appear to have categorical values.  

One question to ask about these data is how many unique levels are there in the columns which are not floating point numbers? To investigate the answer, execute the code in the cell below and examine the results.  

In [ ]:
#for col in HR_data.columns[2:]:  
for col in HR_data.columns:  
    print('\n' + col)
    print(HR_data[col].unique())

There is a variety of information types encoded in these variables.  
1. **Ordinal** integer variables. An ordinal variable has an ordered set of values. In this case, several ordinal variables are numeric. But, `salary` is coded by categories.    
2. Three **binary variables**, coded as $\{0,1 \}$.    
3. One **categorical variable**, `sales`, which are job categories. 
4. Five **numeric variables**, `satisfaction_level`, `last_evaluation`, `number_project`, `average_monthly_hours`, and `time_spent_company` all coded as integers.      

To better understand the numeric variables it is useful to create histograms. Execute the code in the cell below and examine the histograms displayed.     

In [ ]:
for col in ['satisfaction_level','last_evaluation', 'number_project','average_montly_hours','time_spend_company']:
    g = sns.FacetGrid(HR_data, col='left', height=4, aspect=1.5, subplot_kws={'alpha':0.1})
    g.map_dataframe(sns.histplot, x=col)
    plt.show()

None of these variables are remotely close to a Normal distribution. In fact, some of these variables look closer to a uniform distribution.      

Additionally, there are some differences the densities between employees who left and employees who stayed. Optimistically, these differences should help when creating cluster models to understand reasons why employees leave the company.   

### Preparing the data   

With some understanding of the variables, we must now prepare the dataset for the analysis. For clustering algorithms we must restrict ourselves to encoding that preserves our ability to compute the required distance metrics. Still there are a number possible distance metrics and encoding options we can choose from.  

For the examples in this notebook we will use data encoding suitable for Euclidean (L2) and L1 distances. The choice of Euclidean distance, in particular, dictates that we scale numeric features. Additionally, we will minimize the number of binary features.   

As a first step in this process, we must convert the ordinal variable `salary` to a numeric type. The code in the cell below limits the numeric values to the range $[0,1]$ and maintains the order. Execute this code and note the result 

In [ ]:
# Compute float replacements for salary levels
salary_levels = {'low':0.0, 'medium':0.5, 'high':1.0}
salary = [salary_levels[x] for x in HR_data.loc[:,'salary']]

# Delete the object type column and assign the float salary values 
HR_data.drop(columns='salary')
HR_data['salary'] = salary

# Examine the summary
HR_data.loc[:,'salary'].value_counts()

The `sales` variable presents a particular problem. This variable is categorical and there is no sensible ordering. There is no idea solution. In this case we will recode this variable indicating if the position is in sales or something else. This approach avoids creating a larger number of binary dummy variables. Execute the code in the cell below and examine the results. 

In [ ]:
sales = [1.0 if x=='sales' else 0.0 for x in HR_data.loc[:,'sales']]

# Drop the string type column and 
HR_data.drop(columns='sales')
HR_data['sales'] = sales

# Examine the summary
HR_data.loc[:,'sales'].value_counts()

About 1/3 of the employees are in sales, so this coding is reasonable from this point of view. 

Finally, we must normalize the numeric variables. Since the distribution is far from Normal, Z-score normalization is likely to be a poor choice. Instead, we will use the [sklearn.preprocessing.MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html). By default, this scale transforms the scale of each variable to be in the $[0.1]$ range. Such scaling in optimal for uniformly distributed variables, and a good choice for variables that are approximately uniform.  

The code in the cell below applies the Min-Max scalar to the numeric columns. The columns are then zero-centered. Execute this code and examine the results.  

In [ ]:
# Compute normalize values for the columns
normalize_cols = ['satisfaction_level','last_evaluation','average_montly_hours','number_project','time_spend_company']
temp = MinMaxScaler().fit_transform(HR_data.loc[:,normalize_cols])

# Drop columns and add the normalized values to the data frame
HR_data.drop(columns = normalize_cols)
HR_data[normalize_cols] = temp

# Standardize the numeric columns. 
numeric_columns = ['satisfaction_level', 'last_evaluation', 'number_project',
       'average_montly_hours', 'time_spend_company', 'Work_accident', 
       'promotion_last_5years', 'sales', 'salary'] 
#for col in numeric_columns:  
#    HR_data[col] = (HR_data[col] - np.mean(HR_data[col]))/np.std(HR_data[col])
HR_data[numeric_columns] = (HR_data[numeric_columns] - HR_data[numeric_columns].mean())

HR_data.head()

The variable values in the data frame are all in the $[0,1]$ range. No variable will dominate the models simply by having numerically large values. 

At this point, all the variables are numeric. To ensure that all the variables are floating point, execute the code in the cell below. 

In [ ]:
HR_data = HR_data.astype('float64')
HR_data.dtypes

## Visualize the HR Dataset

You will now use the **uniform manifold and projection or UMAP** algorithm to visualize the relationships in the HR dataset. In brief, the UMAP algorithm creates a [** nonlinear low-dimensional projection of high dimensional variable space**](https://en.wikipedia.org/wiki/Nonlinear_dimensionality_reduction). In this case, the projection is on to a 2-dimensional surface in the high-dimensional space, know as a **manifold**. The UMAP algorithm seeks to preserve distance, so that distances on the low-dimensional manifold correspond to the distances in the high-dimensional spaces.   

> **Note:** There is a complication with this particular dataset. Ordinarily, one does not need to perform analysis of the eigenvalues and eigen gap, but here we must do so. To understand this problem execute the code in the cell below to compute and display the eigenvalues of the covariance matrix. This situation is uncommon.   

In [ ]:
covariance = np.cov(np.transpose(HR_data))
eigenvalues = np.linalg.eigvals(covariance)
eigengaps = -np.diff(eigenvalues)
eigengap_ratio = np.divide(eigengaps, eigenvalues[:-1])

nan = [np.nan]
pd.DataFrame({'Eigenvalues':eigenvalues,
             'Eigengaps':np.concatenate((eigengaps,nan),axis=0),   
             'Eigengap_Eigenvalue_ratio':np.concatenate((eigengap_ratio,nan),axis=0)})

Notice that the eigenvalues are in a narrow range. The differences between each ordered pair of eigenvalues is called the **eigengap**. The ratio of the eigengap to the larger eigenvalue is shown in the right column. Notice that this ratio is quite small for the first eigenvalue pair. This result indicates that the eigengap is generally small and the projections to the low-dimensional manifold may not be well defined.          

Next, execute the code in the cell below to compute the UMAP embedding of the HR data frame.    

> **Note:** Because of the eigengap problem identified, you will likely see warning messages from the default eigenvector solver. You can safely ignore these warnings. The UMAP algorithm will converge, but slower than the ideal case.    

In [ ]:
np.random.seed(4365)
reducer = umap.UMAP()
HR_embedding = reducer.fit_transform(HR_data)

print("Success! UMAP fit_transform executed perfectly.")

Indeed, the small eigengap has caused the UMAP algorithm convergence problems. As a result an alternate initialization method has been used. None the less, an low-dimensional embedding has been computed.      

To display the samples projected on the computed manifold, execute the code in the cell below.  

In [ ]:
HR_embedding_df = pd.DataFrame(HR_embedding, columns = ['component1', 'component2'])
HR_embedding_df['left'] = HR_data.loc[:,'left']

fig,ax = plt.subplots(figsize=(6,6))
sns.scatterplot(data=HR_embedding_df, x='component1', y='component2', 
                hue='left', style='left', markers=['o','v'],
                s=5, alpha=0.2, ax=ax, palette={0:'black', 1:'orange'} ) # palette=sns.color_palette("rocket"))
ax.set_xlabel('component 1');
ax.set_ylabel('component 2');
ax.set_title('UMAP projection of HR standardized HR dataset');
plt.show()

The UMAP projection shows good separation between the characteristics of employees who left the company and those that did not. There are a few non-leaving observations that overlap the region of the manifold occupied by the leaving observations.   

At a more detailed level notice the clumping of the observations. For example, the non-leaving observations are in fairly tight groups with considerable separation.    

## K-Means Clustering  

K-means clustering separates a dataset into K clusters of equal variance. The number of clusters, K, is user defined. The basic algorithm has the following steps:
1. A set of K centroids are randomly chosen. 
2. Clusters are formed by minimizing variance within each cluster. This metric is also know as the **within cluster sum of squares** (see further discussion in the section on evaluating clusters). This step partitions the data into clusters with minimum squared distance to the centroid of the cluster. 
3. The centroids are moved to mean of each cluster.    
4. The means of each cluster is computed and the centroid is moved to the mean. 
5. Steps 2, 3 and 4 are repeated until a stopping criteria is met. Typically, the algorithm terminates when the within cluster variance decreases only minimally. 
6. The above steps are repeated starting with a random start of step 1. The best set of clusters by within cluster variance and between cluster separation are retained.  

Since K-means clustering relies only on basic linear algebra operations, the method is fairly scaleable. Out-of-core K-means clustering algorithms are widely used to increase the size of datasets which can be clustered. However, this method assumes equal variance of the clusters, a fairly restrictive assumption. In practice, this criteria is almost never true, and yet K-means clustering still produces useful results. 

### K-means clustering example

Let's try a simple example, applying the k-means algorithm to the prepared HR data. In this case we will use the [sklearn.clustering.KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) package to compute cluster assignments using $k=4$.  

The code in the cell below creates a [k-means cluster object](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans), and computes the cluster assignments with the `fit_predict` method. A frequency table of the cluster assignments by the value of the `left` variable is then displayed. Execute this code and examine the results.     

In [ ]:
nr.seed(4455)
n_clusters=4
HR_data['cluster_assignments'] = KMeans(n_clusters=n_clusters, n_init=10).fit_predict(HR_data)
## Create a frequency table of the number of employees leaving by cluster assignment  
HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index()

Notice that most of the employees who left the company are in a single cluster. Further, there are no assignments to this cluster of employees who did not leave. This appears promising in terms of understanding employees who leave.   

### Analyzing the cluster model

Now that we have a preliminary model we must analyze the results. By itself, the model provides not value. We only add value by providing interpretable and actionable analysis of the results. There are two aspects of the model which we will investigate.    

1. How good is the model in terms of finding compact and well separated clusters. We will explore some metrics for quantitative model comparison shortly. Here will will look at embeddings of the model results to qualitatively evaluate the model.   
2. We can also explore which factors of features in the model are most important in determineing cluster assignment or similarity of cases. In this case we use the results of the cluster model to explore which factors are important in compelling employees to leave the company. This step can provide valuable insight to improve employee retention.   

Or primary tool to investigate and interpret the model results is visualization. Visualization of clustering results can be difficult, but with some effort important insight can be gained. Like most data visualization, a number of ideas must be tested. Most visualizations will not be that useful, but a few will be. Consistent with the prime rule of data mining, **try lots of ideas, fail fast, keep the few ideas that work**.    


**Qualitative check of clustersP:** To understand the quality of the model, we will start by displaying a UMAP projection of the cluster assignments. Such a projection allows one to determine the relationship between the clusters in the manifold space. The manifold projection visualization is intended as as qualitative assessment of the quality of the clusters. Ideally, on the manifold projection the clusters should be both compact and well separated. Execute the code and examine the results.  

In [ ]:
def plot_cluster_assignments(df, 
                             style='left', 
                             s=10, 
                             alpha=0.1, 
                             title='UMAP projection of cluster assignments for HR dataset'):
    
    _,ax = plt.subplots(figsize=(7,6))
    sns.scatterplot(data=df, x='component1', y='component2', 
                    hue='cluster_assignments', style=style, 
                    markers=['o','^'], palette="tab20",
                    s=s, alpha=alpha, ax=ax);
    legend = ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    for handle in legend.legend_handles:
        handle.set_alpha(1.0)  
    ax.set_xlabel('component 1');
    ax.set_ylabel('component 2');
    ax.set_title(title);

## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

## Display the result
plot_cluster_assignments(HR_embedding_df)
plt.show()

There are several observations one can make from this display:    
1. The clustering has generally separated the leaving and non-leaving employees. This observation is consistent with the frequency table discussed previously. From this point of view, the cluster model is successful and might provide insight into the differences between the two populations.    
2. In general, these clusters are not particularly compact. Several of the clusters form multiple groups on the manifold projection. Further, the clusters do not partition the space as we had hoped. Notice that one of the clusters contains significant numbers of employees who both left and stayed.    

**Insight from features determining cluster assignments** The next question is, which of the variables define the differences between clusters? Understanding the answer to this question can provide great insight into the features with important differences between the clusters.    

Intuitively, we would like to identify and explore the variables that have the greatest variation of values between the clusters. The differences in these variable values can help interpret characteristics that define the groups.   

We will search for the variables with the greatest variation between the clusters by the following steps:   
1. Compute the medians of the variable values in each cluster.    
2. Compute the median absolute deviations of the variable medians and sort in descending order.

Now execute the code in the cell below.  

In [ ]:
median_by_cluster = HR_data.groupby('cluster_assignments').median(numeric_only=True)
median_by_cluster

In [ ]:
mad_values = median_by_cluster.apply(lambda x: (x - x.median()).abs().median()).sort_values(ascending=False)
mad_values

> **How to interpret a violin plot:** You may not be familiar with violin plots. Violin plots were introduced in readable paper by [Heintze and Nelson](https://www.stat.cmu.edu/~rnugent/PCMI2016/papers/ViolinPlots.pdf) as a more informative improvement over the familiar boxplot.    
> 
> The violin plot retains the attributes of the familiar [**boxplot**](https://en.wikipedia.org/wiki/Box_plot). On the box plot a dot shows the median of the observations. The inner quartiles are represented by the box. The outer quartiles are shown by the 'whiskers'. 
> 
> The 'violin' on violin plot is a pair of back-to-back [**kernel density estimation (KDE)**](https://en.wikipedia.org/wiki/Kernel_density_estimation) curves. These curves show an empirical estimate of the density of sample values. The KDE curves are normalized to have unit area. This means that even classes with small numbers of observations will have the same total area as curves for curves representing large numbers of samples.    
> 
> Displaying empirical distribution information using the two methods described above methods allows one to visualize quite a bit of information. The box plot retains the ability to show quartiles of the density, in the same familiar manner as a box plot. But, whereas box plots cannot show the properties of multi-modal densities, the violin plot can from the KDE curves. As is commonly done with box plots, violin plots can be arrange side by side for comparing densities of different variables.      

Next, we will display and examine some [violin plots](https://seaborn.pydata.org/generated/seaborn.violinplot.html#seaborn.violinplot). Execute the code in the cell below to display violin plots of variables with the largest median absolute deviations of the median. Hue is used to show the left variable. These variables are chosen since they seem likely to provide some insight into why employees might leave the company. 

In [ ]:
def plot_clusters_by_factor(df, factor='satisfaction_level', type='violin'):
    plt.figure(figsize=(10,4))
    if(type=='violin'):
        ax=sns.violinplot(x='cluster_assignments', y=factor, hue='left', data=df, dodge=True)
    else: 
        ax=sns.boxplot(x='cluster_assignments', y=factor, hue='left', data=df)
    ax.set_title(factor + ' by cluster number')
    plt.legend(bbox_to_anchor=(1.01, 1), borderaxespad=0)
    
plot_columns = [x for x in mad_values[:4].index]
for factor in plot_columns:
    plot_clusters_by_factor(HR_data, factor=factor)    
    plt.show()

Some interesting patterns emerge in these plots:   
1. It is hardly surprisingly that the cluster with the majority of leavers have low median salaries. Notice that one cluster with few leavers have high median salaries.      
2. Not surprisingly, many of the leaving employees have low satisfaction levels as can be seen from the low median for leaving in the cluster with the majority of levers. However, some leavers have high satisfaction level as can be seen by the relatively high median for leaving employees in another cluster. Evidently, satisfaction level alone is not the sole reason for employees leaving.     
3. All clusters with leavers have a high median time since the last review. Notice that these distributions are multimodal.     
4. All leavers in each of the clusters have high monthly hours.     

In summary, we can say that employees leaving the company come in three different groups with distinct characteristics. However all leavers have a long median time since the last evaluation and a high median number of hours worked. This is actionable information that can be used to improve employee retention.     

## Evaluating cluster models

Now that you have created some clustering models, you are likely wondering how can you evaluate these models and perform model selection. There are a number of metrics you can use to evaluate and compare clustering models. However, you should always keep in mind that the best model, should be selected based on the problem you are trying to solve.

### Evaluating sum of squares metrics

One useful metric for clusters is the **within cluster sum of squares** or **WCSS**. Intuitively, clusters should have minimal dispersion and therefore minimal WCSS. The  

$$WCSS = Min \sum_i \sum_{j\ in\ cluster\ i} ||x_j - c_i||^2 \\
where\\
c_i = center\ of\ ith\ cluster\\ 
and\\
||x_j - c_i|| = distance\ between\ data\ x_j\ and\ center\ c_i
$$

We can use WCSS to compare different cluster models based on Euclidean distance measures. Models with smaller SSW have tighter clusters and, therefore smaller WCSS. 


> **Note:** WCSS is also referred to as **inertia**, an analogy with classical mechanics in physics. 


The **between cluster sum of squares** or **BCSS** is a related metric, useful for models using Euclidean distance measures. Whereas WCSS measures how tight the clusters are BCSS is a measure of the separation between the clusters. To compute the BCSS, observe that the **total sum of squares** or **TSS** must equal the sum of the WCSS and BCSS:

$$
TSS = BCSS + WCSS\\
where\\
TSS = \sum_i (x_i - \mu)^2\\
where\\
\mu = mean\ of\ all\ data\ samples
$$

Notice that the TSS is just the variance of all data points. And, BCSS is then just the difference between TSS and WCSS.


> **Note:** The WCSS and BCSS metrics have the concept of the clustering having multivariate-Normal distributions. Therefore, these metrics are strictly only applicable to cluster algorithms using Euclidean distance metrics. This fact means that WCSS and BCSS are not useful metrics for agglomerative clustering. The SC can be computed using various metrics so is more generally applicable to most clustering methods. 

### Silhouette coefficient

Another possible measure for evaluation of clusters is the **silhouette coefficient**, which is computed from the key measures of the clusters:    
- **a:** the mean distance between a sample and all other samples in the cluster, or a measure of **cluster compactness**.   
- **b:** the mean distance between a sample and all other samples in the next nearest cluster, or a measure of **cluster separation**. 

Any distance measure can be used to compute the silhouette coefficient. Therefore, this metric is useful for both Euclidean and non-Euclidean spaces.   

For a clustering model with $N$ samples, the mean silhouette coefficient is then:     

$$S = \frac{1}{N} \sum_{i=N}^N \frac{b - a}{max(a,b)}$$

How can we interpret the silhouette coefficient. The numerator is the difference of the between cluster and within cluster distances. The numerator is normalized by the larger of these two differences. Therefore, we can interpret the silhouette coefficient as the normalized difference of difference of the between cluster and within cluster distances. Given this interpretation, we can say that larger silhouette coefficient is generally better. But be careful, large silhouette coefficients can sometimes arise from fragmentation of the data into a large number of generally meaningless clusters.    

### Calinski-Harabasz Index     

The Calinski-Harabasz score is a **degree of freedom adjusted** variance ratio. The variance is taken from the trace of the BCSS and WCSS matrices. The trace of a matrix is the sum of the diagonal terms. For BCSS and WCSS the trace is the sum of the cluster-wise variances.   

The Calinski-Harabasz score for the $ith$ cluster is writen:    

$$ch_i = \frac{tr(BCSS)\ n_e - k}{tr(WCSS)\ k-1}$$     
Where:    
$n_e =$ is the total number of observations.   
$k =$ the number of clusters.   

The Calinski-Harabasz index is the mean of the scores for each of the clusters:   

$$\bar{CH} = \frac{1}{k} \sum_{i=1}^k ch_i$$   

A **larger Calinski-Harabasz Index** indicates well separated and compact clusters.  

### Davies-Bouldin Index   

The Davies-Bouldin index is a measure cluster compactness and separation. We can write the Davies-Bouldin score between cluster $i$ and $j$: 

$$s_{i,j} = \frac{S_i + S_j}{D_{i,j}}$$

Where dispersion and distance can be measured with mot any distance metric,  
$S_i$ is the the dispersion (inertia) of cluster $i$.      
$D_{i,j}$ is the distance between the centroids of cluster $i$ and cluster $j$.    

The Davies-Bouldin index is the mean of the maximums of the scores for each cluster with respect to the other clusters:   

$$R_i = \max_{j}(s_{i,j})$$
And the Davies-Boldin score is the mean of the maximums for each cluster:       
$$\bar{R} = \frac{1}{k} \sum_{i=1}^k R_i$$   

The **small Davies-Bouldin index** indicates that the clusters are compact and well separated.  

### Finding the number of clusters

We have investigated our first k-means cluster model with $k = 4$. The next question is, how many clusters ($k$) are optimal. As we are dealing with unsupervised learning models the answer is generally not obvious. 

One approach to determining the number of clusters is to plot several metrics against the number of clusters. In this case you will use WCSS, silhouette index, Davies-Bouldin index and Calinski-Harabasz index. The code in the cell below creates just such a plot by the following steps:
1. Any existing cluster assignments are removed from the data frame. It is critical that assignments from other models not bias the training of the new models. 
2. Lists are defined to save the metrics computed for each value of $k$.
3. A loop iterates over the specified range of $k$ values. 
4. For each value of $k$ **inertia** or WCSS, silhouette index, Davies-Boldin index and Calinski-Harabasz index are computed and saved. The `inertia` attribute of the model, along with the [silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html), [Davies-Bouldin index](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.davies_bouldin_score.html) and [calinski_harabasz_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.calinski_harabasz_score.html) functions are used to compute these metrics.  
5. An array of plots are displays WCSS, silhouette index, Davies-Bouldin index and Calinski-Harabasz index vs. the number of clusters. 

Execute this code and examine the result.

In [ ]:
def cluster_search_kmeans(df, nclusts=(2,21), n_init=10):
    ## If there are cluster assignments in the data frame remove them. 
    if 'cluster_assignments' in df.columns: df.drop(columns='cluster_assignments', inplace=True)
    WCSS=[]
    silhouette=[]
    CH_index = []
    DB_index = []

    ## Iterate over values of k
    for n in range(nclusts[0],nclusts[1]+1):   
        temp_model = KMeans(n_clusters=n, n_init=n_init).fit(df)
        WCSS.append(temp_model.inertia_)
        assignments = temp_model.predict(df)
        silhouette.append(silhouette_score(df, assignments))
        CH_index.append(calinski_harabasz_score(df, assignments))
        DB_index.append(davies_bouldin_score(df, assignments))
    _, ax = plt.subplots(2,2, figsize=(8,8))    
    ax = ax.flatten()
    ax[0].plot(range(nclusts[0],nclusts[1]+1),WCSS)   
    ax[0].set_xlabel('Number of clusters')
    ax[0].set_ylabel('WCSS')
    ax[1].plot(range(nclusts[0],nclusts[1]+1),DB_index)   
    ax[1].set_xlabel('Number of clusters')
    ax[1].set_ylabel('Davies-Bouldin Index')
    ax[2].plot(range(nclusts[0],nclusts[1]+1),silhouette)   
    ax[2].set_xlabel('Number of clusters')
    ax[2].set_ylabel('Silhouette Index')
    ax[3].plot(range(nclusts[0],nclusts[1]+1),CH_index)   
    ax[3].set_xlabel('Number of clusters')
    ax[3].set_ylabel('Calinski Harabasz Index')
 
nr.seed(9966)
cluster_search_kmeans(HR_data)    
plt.show()

The slope of the WCSS curve shows no particular rapid change or 'knee'. This situation is know to occur frequently and limits the use of this method. The silhouette coefficient has one clear peak at $k=5$. There are two close minimum of the Davies-Boldin at $k=5$ and $k=9$. The Calinski-Harabasz index has a peak at $k=3$, with the value at $k=5$ is nearly as large. There is a bit of a conflict in these statistics. A compromise at $k=5$ is likely to be the best choice.  

### Exploring model results

> **Exercise 3-1:** In the cell below create and execute code to do the following: 
> 1. Make sure you first remove the `cluster_assignments` column from the data frame if present. 
> 2. Use the `%time` IPython magic as a prefix of the line of code creating the cluster model. 
> 3. Compute an k=7 cluster model for the HR dataset, using the ` n_init=10` argument for 10 random starts. using the `fit_predict` method, and assign the clusters from the model to a 'cluster_assignments' column in the HR data frame.  
> 4. Display a frequency table of cluster assignments by value of the `left` variable

In [ ]:
nr.seed(7722)
## Put your code below


> Examine the frequency table and answer the following questions:  
> 1. Are there clusters which contain the majority of records for the employees who left the company?  
> 2. Do the foregoing clusters contain any records for employees who stayed with the company?    
> 3. Given your answers to the foregoing questions, do you think the model can provide useful insight into characteristics of customers who leave the company?     
> **End of exercise.**


> **Answers:**    
> 1.          
> 2.     
> 3.     

> **Exercise 3-2:** To further evaluate the model you have computed, create execute the code in the cell below to display the UMAP plot using the `plot_cluster_assignments` function.   

In [ ]:
## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

## Display the projection of the cluster assignments
plot_cluster_assignments(HR_embedding_df)
plt.show()

> Examine this plot and answer the following questions: 
> 1. Has this cluster model better separated the leavers and non-leavers compared to the k=4 model? 
> 2. Do the clusters which contain the majority of records of employees who have left the company appear in reasonably tight clusters on the UMAP projection?     

> **Answer:**
> 1.    
> 2.     

> Next, you will display and examine some violin plots. Create and execute the code in the cell below to display violin plots of variables with the largest median absolute deviation of the medians. Use the `find_plot_columns` function to identify the variables with the largest median absolute deviation of the medians. Hue is used to show the left variable. These variables are chosen since clusters containing records of employees who left the company are in a reasonably small range along these axes in the scatter plot matrix. 

In [ ]:
def find_plot_columns(df, threshold = 0.001):
    cluster_stats = df.groupby('cluster_assignments').agg(['median'])
    mad_values = cluster_stats.apply(lambda x: (x - x.median()).abs().median()).sort_values(ascending=False)
    mad_values = mad_values[mad_values > threshold]
    return [x[0] for x in mad_values.index]

## Put your code below
for factor in find_plot_columns(HR_data):
    plot_clusters_by_factor(HR_data, factor=factor)   
    plt.show()

> We can make some inferences from these results. Keep in mind the variable plots displayed are in descending order of median absolute deviation of the medians. Answer the following questions:  
> 3. What variable values differentiate the majority of leaving employees from non-leavers and how might this observation be helpful in improving employee retention?   
> 4. Which variables differentiate between the two clusters containing the majority of the leaving cases and how might these observations improve employee retention?     
> **End of exercise.**

> **Answers:**    
> 3.    
> 4.     

## Mini-Batch K-Means

The mini-batch K-means algorithm follows the same steps as the conventional (Batch) K-means algorithms. However at each step, a randomly selected mini-batch is used to update cluster centers and cluster assignments. This algorithm iterates through mini-batches until convergence. 

> **Exercise 3-3:** You will now create and benchmark code to implement the mini-batch K-means algorithm. You will use the [sklearn.clustering.MiniBatchKMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.MiniBatchKMeans.html) function. Use the arguments `n_clusters=7`, `batch_size=512`, and `n_init=10`, using the `fit_predict` method, and assign the clusters from the model. Use the `%time` directive to capture the execution time. When the cluster calculation is completed, generated and print the frequency table of the cluster assignments.       

> **Note:** If you are on Windows, you will likely see a warning concerning a known memory leak in the MiniBatchKMeans function. Since you only runing this function once on a small dataset, it is safe to ignore this warning. 

In [ ]:
nr.seed(7722)
## Put your code below



> Examine your results and answer these questions in one or a few sentences:  
> 1. Compare the execution time of the full batch k-means algorithm to the execution time for the batch K-means. How does the run-time of the mini-batch algorithm compare to the full batch k-means algorithm and is this expected?   
> 2. Compare the number of assignments of cases of leaving employees assigned to clusters with predominantly cases of non-leaving employees to the same labeled cases for the k=5 full batch algorithm. What can you say about the mini-batch solution vs. the full batch solution and why is this behavior expected?       
> **End of exercise.**

> **Answers:**    
> 1.          
> 2.               

## Introduction to Hierarchical Clustering

Another widely used form of clustering uses hierarchical modes. These models attempt to divide or partition the data following a hierarchical sequence. 

Hierarchical clustering models produce a tree-like organization of the data into a hierarchy of clusters. At the root of the tree, all data cases are in one large cluster. The leaves of the tree each have a cluster with a single data case.

Hierarchical clustering models generally use one of two approaches. 

1. **Agglomerative clustering** works from the leaves of the tree toward the root in the following way.
  - All data cases start in their own cluster. 
  - Pairs of clusters are merged to their nearest neighbors over several iterations. 
  - The second step is repeated until there is one cluster at the root of the tree. 
2. **Divisive clustering** works from the root of the tree toward the leaves in the following way.
  - All data cases start in a single large cluster. 
  - Clusters are split into two parts in a way that maximizes the distance (or dissimilarity) between the clusters. 
  - The second step is repeated until there the leaves of the tree, each with a single data case, is encountered. 
  
  
With the above descriptions of the algorithms in mind, we need to discuss how distance is measured. 

First, we select a **distance metric** to compute the distance between two individual data points. We have already encountered some of the most widely used metrics, **Euclidean** and **Manhattan**. There are a great many other metrics one can choose. As an example, execute the code in the cell below to generate a partial list of [metrics supported by Scikit-Lean](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.DistanceMetric.html).    

In [ ]:
from sklearn.metrics.pairwise import distance_metrics

# Get the dictionary of available distance metrics and display them
available_metrics = distance_metrics()
print("Available basic distance metrics in scikit-learn:")
for metric_name in available_metrics.keys():
    print(f"- {metric_name}")

Since clusters are typically made up of multiple points we need a way to combine the distances between the points in clusters. We do this through **linkage functions**. There are several commonly used linkage functions. For a distance metric value between two points, $a$ in the one cluster, and $b$ in the other cluster, $D(a,b)$, we can define some common linkage functions:

1. **Ward's method** is a linkage method that uses a minimum variance criteria to select the pairs of smaller clusters to link. Since this method is based on variance, it is only defined for Euclidean spaces.     
2. **Maximum or complete linkage** is the largest value of the distance metric between any pairs of points in the two clusters. 
$$= Max \big( D(a,b) \big)$$
3. **Minimum or single linkage** is the smallest value of the distance metric between any pairs of points in the two clusters. 
$$= Min \big( D(a,b) \big)$$
4. **Mean or average linkage** is the average of the distance metrics between all pairs of points in the two clusters.
$$= \frac{1}{N_{ab}} \sum D(a,b)\\ 
where\\
N_{ab}\ is\ the\ count\ of\ pair-wise\ distances$$
5. **Centroid linkage** is the distance metric between the centroids between the two clusters. 
$$= |c_1 - c_2|\\
where\\
c_1, c_2\ are\ centroids\ of\ clusters\ 1\ and\ 2$$

As you can imagine, the choice of distance metric and linkage function can significantly change the clustering relationships a model finds. There are some restrictions on this choice one should always keep in mind. Ward's method is only optimal for Normally distributed data and centroid linkage assumes a Euclidean space. Complete-linkage, single linkage and average linkage are all applicable to non-Euclidean spaces. In other words, the linkage method selected must match the properties of the distance metric being used and vice versa.      

### Agglomerative clustering example   

With the foregoing theory in mind, its time to try an example using the HR dataset. We have already explored k-means clustering on this dataset. Recall that the k-means method is only defined for the Euclidean distance metric.  

The agglomerative clustering algorithm, like all hierarchical clustering algorithms, Allows the choice of both a linkage method and a range of distance metrics. In this first example we will use complete linkage which uses the maximum distance between any two points within two clusters. We will also use the Manhattan (L1) distance metric.  

The code in the cell below creates and fits an agglomerative cluster model and then displays the resulting frequency table. Execute this code and examine the results.

In [ ]:
nr.seed(2356)
if 'cluster_assignments' in HR_data.columns: HR_data.drop(columns='cluster_assignments', inplace=True)
model_agglomerative =  AgglomerativeClustering(n_clusters=4, linkage='complete', metric='manhattan',
                                              compute_full_tree=False)   
HR_data['cluster_assignments'] = model_agglomerative.fit_predict(HR_data)
HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index(axis=0, level=0)

There is one cluster containing most of the records of the employees who left the company.   

Execute the code in the cell below to create the scatter plot matrix of some of the variables with the hue showing cluster assignments. 

In [ ]:
## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

#plot_cluster_assignments(HR_data)
plot_cluster_assignments(HR_embedding_df)
plt.show()

The records assigned to the cluster with the employees who left the company fall in several groups on the graphs. While it is good most of these records are in one cluster, interpreting the characteristics of these employees in a meaningful way will be difficult at best.      

### Evaluating number of clusters

Given the foregoing results it seems entirely possible that the a different number of clusters might provide better insight into to structure of the data. But how can we evaluate the clusters? With **non-Euclidean** we cannot use the WCSS. We need alternatives. There are several possibilities     

A simple metric is **Maximum diameter of clusters**. As more clusters are used in a model the clusters become smaller. But, as the model becomes complex, the reduction in the size of the clusters diminishes. The distance must be measured using the clustering distance metric, which can be non-Euclidean. The maximum diameter of cluster, $C_i$, is:  

$$diameter(C_i) = max_{j,k \in C_i} d(x_j,x_k)$$    

In addition to maximum cluster diameter, the silhouette coefficient is a useful metric for non-Euclidean distance metrics.

> **Exercise 3-4:** 
> You will complete the code in the cell below so that it performs the following operations:    
> 1. The `evaluate_agglomerative_clusters` function Iterate over each cluster and performs the following operations.    
>  - Delete any previous cluster assignment column.   
>  - Create a new temporary model.  
>  - Cluster assignments are made with the predict method.  
>  - Compute the silhouette coefficient with the `silhouette_score` function and append the result to the `silhouette_coefficients` list.   
>  - The `find_max_diameter` function is called and the value is appended to the `max_diameters` list. 
> 2. The `find_max_diameter`function iterates over all cluster assignments doing the following: 
>   - Create a temporary data frame containing only samples assigned to the particular cluster (ith cluster), make sure to drop the `cluster_assignments` column. 
>   - Using the temporary data frame compute the maximum diameter and append it to the 'max_diameters' list. Use the [sklearn.metrics.pairwise_distances](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise_distances.html) function and the [numpy.amax](https://numpy.org/doc/stable/reference/generated/numpy.amax.html) function. 
>   - The maximum of the maximum is returned.      
> 3. Plots the maximum of the diameter of the clusters for each value of $k$.   
> Execute the code and examine the results. Since hierarchical clustering is computationally intensive, execution of this code may take some time.  

In [ ]:
def find_max_diameter(df, metric='manhattan'):
    max_diameters = []
    ## Put your code below




def evaluate_agglomerative_clusters(df, metric='manhattan', linkage='complete', nclusts=(4,18)): 
    silhouette_coefficients = []
    max_diameters = []
    for k in range(nclusts[0],nclusts[1]+1):
        ## Put your code below
        ## First compute the cluster assignmenets for the number of clusters, k
        

        
        
        ## Compute and append the silhouette coefficeint to the list 
        
        
        
        ## Find the max diameter of the clusters
        ## Add the cluster assignment column to the data frame
    

    ## Plot the results     
    _, ax = plt.subplots(1,2, figsize=(12,5))    
    ax[0].plot(range(nclusts[0],nclusts[1]+1), max_diameters);
    ax[0].set_xlabel('Number of clusters')
    ax[0].set_ylabel('Maximum cluster diameter')
    ax[0].set_title('Maximum cluster diameter vs. number of clusters')
    ax[1].plot(range(nclusts[0],nclusts[1]+1), silhouette_coefficients);
    ax[1].set_xlabel('Number of clusters')
    ax[1].set_ylabel('Silhouett Coefficients')
    ax[1].set_title('Silhouett coefficient vs. number of clusters')
    return pd.DataFrame({'NumberClusters':range(nclusts[0],nclusts[1]+1),
                         'ClusterDiameter':max_diameters ,
                         'SilhouetteCoefficient':silhouette_coefficients})  

np.random.seed(6745)
evaluate_agglomerative_clusters(HR_data)    
plt.show()

> Examine the plot. These curves are not smooth. The silhouette coefficient peaks at 12 clusters. The maximum cluster diameter is still monotonically decreasing at 12 clusters with a possible break in slope at 13 clusters. These observations mean that the trade-off between compact clusters and cluster separation is approximately optimal at 13 clusters.     
>
> Next, in the cell below create and execute code to do the following:   
> 4. Delete the cluster assignment column if one is present.   
> 5. Compute the 12 cluster model with complete linkage and the Manhattan distance metric, using the `fit_predict` method, and assign the clusters from the model. Prefix the line of code calling the `fit` method with the `%time` directive.     
> 6. Display a frequency table of the `left` variable by cluster assignment.  

In [ ]:
nr.seed(2323)
## Put your code below


## Display the frequency table
HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index(axis=0, level=0)

> Answer the following questions in one or a few sentences.   
> 1. Are the majority of the records of employees who left the company in clusters without employees who did not leave the company and what does this tell you about the ?   
> 2. Given the foregoing observation does this model appear to identify some structure in the data which might provide insight into employees leaving the company and what is it?    


> **Answers:** 
> 1.      
> 2.    

> To further evaluate this model, create and execute the code in the cell below to UMAP plot and examine the results.    

In [ ]:
## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

## Put your code below
plot_cluster_assignments(HR_embedding_df)
plt.show()

> 3. Examine this plot displaying 13 clusters. Are the clusters containing the employees leaving displayed in the plots in tight (small) clusters and well separated clusters?     

> **Answer:** 3.    

> Next, you will display and examine some violin plots. Create and execute the code in the cell below to display violin plots of variables with the largest median absolute deviation of the medians. Use the `find_plot_columns` function to identify the variables with the largest median absolute deviation of the medians. Hue is used to show the left variable. 

In [ ]:
## Put your code below




> These results look similar to the those from the k=5 k-means algorithm. However, in detail there are some differences representing the different algorithms used. Examine these plots. For the three clusters with large number of leavers describe differences in the variable values that differentiate these clusters.    
> **End of exercise.**

> **Answer:** The differences in these variables are:     
> a.       
> b.      
> c.           

## Spectral Clustering   

Spectral clustering is a graph-based clustering algorithm. In summary, the algorithm uses the following steps:  

1. Create an undirected graph of the data samples. This graph can be fully connected or use only nearest-neighbors. The edge weights are the similarity between the samples. An association matrix is created from this graph.  
2. The graph Laplacian matrix is computed. 
2. An eigen-decomposition of the graph Laplacian is performed. The k eigenvectors corresponding to the smallest nonzero eigenvalues define the **spectrum** of the graph.   
3. A clustering algorithm is applied to the eigenvectors. Typically, a k-means algorithm is used.  

> **Exercise 3-5:** You will now apply the [sklearn.cluster.SpectralClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.SpectralClustering.html) to compute a spectral clustering model.  
> 1. Start with the original data encoding (not the cosine similarity encoding) and make sure there is no cluster assignment column.   
> 2. Fit a model using `[sklearn.cluster.SpectralClustering` with the following arguments; `assign_labels='discretize'`, `n_clusters=10`, `affinity='nearest_neighbors'`, `n_neighbors=50`, and `random_state=0`, and using the `fit_predict` method. Prefix your call to `SpectralClustering` with the `%time directive`. 
> 3. Compute and print the frequency table of the cluster assignments.      
> **Note on algorithm:** In this case the spectral clustering algorithm is using nearest neighbor affinity, rather than a fully connected graph. This choice has several consequences. First, the affinity matrix is sparse, making the calculation of the eigenvectors considerably faster, although it may take some time to execute this code as it is. Second, the `SpectralClustering` function may raise a warning that the embedding is an approximation, since the graph is not fully connected.    
> **Note:** Executing this code may take some time. 

In [ ]:
nr.seed(7788)
## Put your code below


## Display the frequency table
HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index(axis=0, level=0)

> Answer the following questions in one or a few sentences.   
> 1. Are the majority of the records of employees who left the company in clusters without employees who did not leave the company?   
> 2. Given the above, does this model appear to identify some structure in the data which might provide insight into employees leaving the company?    

> **Answers:**     
> 1.       
> 2.     

> To further evaluate this model, create and execute the code in the cell below to display the UMAP projection plot and examine the results. 

In [ ]:
## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

## Put your code below
plot_cluster_assignments(HR_embedding_df)
plt.show()

> 3. Examine this plot displaying 10 clusters. Are the clusters containing the employees leaving displayed in the plots in tight (small) clusters?     

> **Answer:** 3.            

> Next, execute the code in the cell below to display violin plots to further explore the relationships in the clusters. Do this for satisfaction level, number of projects, and average monthly hours by cluster assignment.  

In [ ]:
for factor in find_plot_columns(HR_data):
    plot_clusters_by_factor(HR_data, factor=factor)   
    plt.show()

> In detail these results look somewhat different from those from the k=5 k-means algorithm. Consequently, one can get a new perspective on the problem of employees leaving the company:   
> 4. What evidence is noticeable in these clusters that employees with low satisfaction are likely to leave the company?      
> 5. What is the evidence that employees high or low numbers of hours worked are likely to leave the company?            
> **End of exercise.**

> **Answers:**     
> 4.                 
> 5.     

## Density Clustering with OPTICS

The OPTICS algorithm is an example of a density clustering algorithm. The algorithm constructs a graph of the data samples. Samples in the dense core of the clusters are connected by undirected edges. Samples on the periphery of the clusters are corrected by directed edges. Samples too far from any cluster are unconnected to any cluster. Unlike the other models you have worked with in this assignment, OPTICS does not require the specification of the number of clusters. The algorithm determines the number of clusters from the number of dense regions found.    

> **Exercise 3-6:** You will now apply the OPTICS algorithm to the HR data set.      
> 1. Make sure there is no cluster assignment column.   
> 2. Fit a model using [sklearn.cluster.OPTICS](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.OPTICS.html) with the following arguments; `p=2`, `min_samples=250`. Use the `fit` method, not fit_predict. Make sure you name your model object `optics_model`. We will use the cluster model object a bit later. Prefix you call to `Optics` with the `%time` directive.    
> 3. Extract the cluster labels from the model object, which are the `.labels_` attribute of the object. 
> 4. Compute and print the frequency table of the cluster assignments.      
> **Note:** Executing this code may take some time. 

In [ ]:
nr.seed(4512)
## Put your code below




## Display the frequency table
HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index(axis=0, level=0)

> Answer the following questions in one or a few sentences.   
> 1. Notice the cluster with label `-1`. These are samples that are too far from any cluster code to be assigned a label. How do you think these unassigned cases affect the interpretation of the model?   
> 2. Are the majority of the records of employees who left the company in clusters without employees who did not leave the company?   
> 3. Given the above, and the number of clusters, does this model appear to identify some structure in the data which might provide insight into employees leaving the company?    

> **Answers:**      
> 1.       
> 2.    
> 3.    

> To further evaluate this model, execute the code in the cell below and examine the resulting embedding display. 

In [ ]:
## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

# Filter out the unassigned data points 
HR_temp = HR_embedding_df[HR_data.cluster_assignments > -1]

# Display the results 
plot_cluster_assignments(HR_temp)
plt.show()

> 4. Are most of the clusters with employees who left relatively small or compact?      

> **Answer:** 4.             

> To further evaluate this model, execute the code in the cell below and examine the resulting violin plots.  

In [ ]:
for factor in find_plot_columns(HR_data):
    plot_clusters_by_factor(HR_data, factor=factor)   
    plt.show()

> 5. Examine the foregoing plots, particularly the observations that did not fit into any cluster. What observations can you make about these observations and how do you observations explain why these observations did not fit with any cluster.                   

> **Answer:** 5.    

> Finally, let's examine the reachability of the data samples for this density clustering model. The code in the cell below does the following:  
> 1. Extracts the ordered reachability and labels from the model object.    
> 2. Remove the cases where the sample is not a member of any cluster. 
> 3. Plot the reachability, using cluster label as the hue argument. 

In [ ]:
## Extract the ordered reachability and cluster labels
reachability = optics_model.reachability_[optics_model.ordering_]
labels = optics_model.labels_[optics_model.ordering_]

## Remove the cases where the sample is not in a cluster
reachability = reachability[labels > -1]
labels = labels[labels > -1]

## Plot the graph
plt.figure(figsize=(12,5))
ax = sns.lineplot(x=range(reachability[1:].shape[0]), y=reachability[1:], hue=labels[1:], legend="full", palette="Paired")
ax.set_ylabel('Epsilon');
ax.set_xlabel('Observation');
ax.set_title('Reachability plot');
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.);
plt.show()

> What conclusions can we make from this graph? An important observation is that there is a wide range of reachability values for all the clusters. Further, the reachability of some clusters is much higher than for others. Answer these questions:   
> 6. Do clusters have different densities of samples, and is this is expected and desirable given the complexity of the data?  
> 7. Most of these clusters must have some chain effect. This is evidenced by the fact that the range of reachability values varies greatly within most of the clusters. What does this tell you about how compact and well-formed the clusters are?    
> **End of exercise.**    

> **Answers:**     
> 6.           
> 7.    

### OPTICS as a Hierarchical Model

In the foregoing example we have only looked at a flat OPTICS model. However, OPTICS is an hierarchical model and we will now investigate these properties. The code in the cell below iterates through a number of the values of $max\ \epsilon$, which restricts the maximum distance between two points to be considered in the same neighborhood. Steps in this process are:   
1. An OPTICS model is instantiated and fit using the value of $max\ \epsilon$.     
2. A table of cluster assignments is printed.
3. A UMAP projection of the cluster assignments is displayed.    
4. A reachability plot is displayed.

Execute the code and carefully examine the result.  

In [ ]:
def optics_explore(eps_list, df, min_samples=250):   
    for eps in eps_list:
        if 'cluster_assignments' in df: df.drop(columns='cluster_assignments', inplace=True)
        optics_model = OPTICS(max_eps=eps, min_samples=min_samples, p=2).fit(df)
        df['cluster_assignments'] = optics_model.labels_
        
        print("\nWith eps = " + str(eps))
        print(df.loc[:,['cluster_assignments','left']].value_counts().sort_index(axis=0, level=0))

        ## Extract the ordered reachability and cluster labels
        reachability = optics_model.reachability_[optics_model.ordering_]
        labels = optics_model.labels_[optics_model.ordering_]

       # temp = df.loc[df.cluster_assignments > -1]
      #  print(temp.columns)
      #  plot_cluster_assignments(temp,
      #                          title="UMAP projection with eps = " + str(eps))
      #  plt.show()

        ## Remove the cases where the sample is not in a cluster
        reachability = reachability[labels > -1]
        labels = labels[labels > -1]

        ## Plot the graph
        plt.figure(figsize=(12,5))
        ax = sns.lineplot(x=range(reachability[1:].shape[0]), y=reachability[1:], hue=labels[1:], legend="full", palette="Paired")
        ax.set_ylabel('Epsilon');
        ax.set_xlabel('Observation');
        ax.set_title(f'Reachability plot for eps_max= {eps}');
        plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.);
        plt.show()

       

eps_list = [0.7,0.6,0.5,0.4,0.3,0.2,0.1]
optics_explore(eps_list, HR_data)
plt.show()

> **Exercise 3-7:** Carefully examine how the clusters and the reachability plot as the $max \epsilon$ decreases and answer these questions in one or a few sentences.
> 1. How does the number of outliers not assigned to the clusters change as $max\ \epsilon$ changes?    
> 2. How does the number of clusters and the size of the clusters change as $max\ \epsilon$ changes ?
> 3. Explain how the change in $max\ \epsilon$ explains your observations used to answer the preceding two questions.    
> 4. Explain how the change in $max\ \epsilon$ traverses the cluster hierarchy.  

> **Answers:**
> 1.    
> 2.    
> 3.   
> 4.    

## Hierarchical Clustering with HDBSCAN

The [HDBSCAN algorithm](https://hdbscan.readthedocs.io/en/latest/index.html#) is a sophisticated and efficient hierarchical density clustering algorithm. The algorithm has a number of hyperparameters with two being of particular importance:   
1. `min_samples` determines the minimum number of samples required to consider the neighborhood of a point as 'core'. If this argument is left as `None`, min_samples is set to min_cluster_size.     
2. 'min_cluster_size' does exactly what the argument name implies, limiting cluster formation to clusters with at least this number of points single-linked.
3. `cluster_selection_epsilon` clusters with reachability distance less than this value between the nearest points will be merged. Setting this hyperparameter can prevent the formation of many small clusters.      

### Example of HDBSCAN    

The code in the cell below builds an HDBSCAN model for the HR data. A frequency table for the cluster assignments and unassigned values, or outliers is then printed followed by displaying a UMAP projection of the cluster assignments. Execute the code and examine the results.    

In [ ]:
min_cluster_size = 200
min_samples = 400
if 'cluster_assignments' in HR_data.columns: HR_data.drop(columns='cluster_assignments', inplace=True)
#%time HDBSCAN_cluster = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples).fit(HR_data)
HDBSCAN_cluster = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples).fit(HR_data)

HR_data['cluster_assignments'] = HDBSCAN_cluster.labels_
print(HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index())

## Add the cluster assignments to the embedding data frame
HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

## Display the results
plot_cluster_assignments(HR_embedding_df)
plt.show()

To get a feel for the points that are not assigned to any cluster, the outliers, we can plot the probability of an assignment for each point. To display this plot, execute the code in the cell below.   

In [ ]:
sns.histplot(data=HDBSCAN_cluster.probabilities_);
plt.show()

The points not assigned to clusters have a probability of 0, whereas points unambiguously assigned have probability 1. Other points are assigned with probabilities less than 1, but greater than 0.     

In theory, we could display a dendrogram of the HDBSCAN cluster hierarchy. However, for all the very small datasets our ability to interpret or even see the result will be limited at best. HDBSCAN has a useful method that allows plotting a condensed cluster hierarchy. The vertical axis of the plot is parameterized by the inverse of reachability distance, $\epsilon$, or $\lambda = 1/\epsilon$. The width of the cluster marker indicates the size of the cluster, decreasing as outlying points are removed removed from the cluster with decreasing $\lambda$. Execute the code in the cell below and examine the results.     

In [ ]:
HDBSCAN_cluster.condensed_tree_.plot();
plt.show()

There is one last step for exploring the cluster hierarchy. The `select_clusters=True` argument allows us to view which of the clusters shown are in the final flat cluster assignments. To see this display, execute the code in the cell below.  

In [ ]:
HDBSCAN_cluster.condensed_tree_.plot(select_clusters=True,
                               selection_palette=sns.color_palette('deep', 8));
plt.show()

### HDBSCAN as an Hierarchical Model

The HDBSCAN algorithm computes a cluster hierarchy. Exploring this hierarchy provides a great deal of insight into the relationships in a dataset.    

> **Exercise 3-8:** In this exercise you will explore some of the hierarchical properties. The code in the cell below iterates through a number of values for the `min_samples` and `min_cluster_size` hyper parameters for clustering the HR dataset. For each pair of hyperparameters the frequency table and diagnostic plots are displayed. Execute this code and examine the results.            

In [ ]:
min_cluster_sizes = [100, 100, 200, 200, 300, 300, 400, 400]
min_sampless = [100, 200, 200, 400, 300, 600, 400, 800]

for min_cluster_size, min_samples in zip(min_cluster_sizes, min_sampless):
    print(f"\n\n\n\n With min_cluster: {min_cluster_size}  min_samples: {min_samples}")
    if 'cluster_assignments' in HR_data.columns: HR_data.drop(columns='cluster_assignments', inplace=True)
    HDBSCAN_cluster = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples).fit(HR_data)

    HR_data['cluster_assignments'] = HDBSCAN_cluster.labels_
    print(HR_data.loc[:,['cluster_assignments','left']].value_counts().sort_index())
    
    ## Add the cluster assignments to the embedding data frame
    HR_embedding_df['cluster_assignments'] = HR_data.loc[:,'cluster_assignments']

    ## Display the results
    plot_cluster_assignments(HR_embedding_df)
    plt.show()

    HDBSCAN_cluster.condensed_tree_.plot(select_clusters=True,
                               selection_palette=sns.color_palette('deep', 8));
    plt.show()

> Examine the results for each pair of hyperparameters and answer the following questions in one or a few sentences.
> 1. How does the number of unclustered points change with the hyperparameter values, and why?
> 2. How does cluster size change  with the hyperparameter values, and why?
> 3. How does cluster size change with the value of $\lambda$ and why?
> 4. In several cases, a cluster is formed higher in the hierarchy, at say $\lambda = 1$, with the branches of these nodes not forming clusters. Explain this behavior.
> 5. Closely related to the previous question, notice that for many of the HDBSCAN models the clusters form at different values of $\lambda$ or $\epsilon$. Explain why this behavior is expected and why it occurs.
> 6. How Does the cluster formation process in the hierarchical HDBSCAN model differ from the flat cluster formation of the hierarchical OPTICS model and why does this different make cluster formation with HDBSCAN more dynamic?        

> **Answers:**
> 1.    
> 2.    
> 3.    
> 4.    
> 5.    
> 6.    

## K-means with PCA    

Clustering algorithms, like nearly all machine learning algorithms, struggle when faced with very high dimensional data. As an example, we will work on clustering the genetic data for patients with either Crohn's disease and colitis. This is difficult problem with data from only 97 patients and over 10000 gene expressions. Further, there is class imbalance, with Crohn's disease being the minority case.    

The projection of gene data features onto the orthogonal PCA space, can help with creating a better cluster model. In this case, we will create a k-means cluster model using the first 40 components. 

Execute the code in the cell below to load the data and display the data frame.      

In [ ]:
gene_data = pd.read_csv('../data/ColonDiseaseGeneData-Cleaned.csv')
labels = gene_data.loc[:,'Disease State']
gene_data = gene_data.drop('Disease State', axis=1)

## Normalize the columns 
gene_data = (gene_data - gene_data.mean(axis=0)) / gene_data.std(axis=0)

## Display the results 
print('Shape of the data array = ' + str(gene_data.shape))
print(gene_data.head())

As a first step, we need to determine how many clusters there should be. To find out the code in the cell below does the following:     
1. Computes the projections for the first 40 principle components of the gene data.     
2. Applied the k-means clustering algorithm to explore models from $k=2$ to $k=10$.       

Execute the code in the cell below and examine the resulting plots.  

In [ ]:
n_components = 40  
gene_pca = PCA(n_components=n_components)
gene_pca_transform = gene_pca.fit_transform(gene_data)
gene_pca_transform = pd.DataFrame(gene_pca_transform[:,:n_components], columns=[str(i) for i in range(n_components)])
    
np.random.seed(9686)
cluster_search_kmeans(gene_pca_transform, nclusts=(2,10)) #, label_column='Disease')    
plt.show()

In this case, while the dimensionality of the data is high (10497), there are only 97 cases. Therefore we will prefer models with few clusters. Models with a larger number of clusters risk fragmentation, or over-fitting. As a result of the small sample we limit our exploration to 20 clusters maximum.     

There is no reason to believe that the differences in the response of a few genes specific to the persons' condition is sufficient to separate these cases. There may be other genetic similarities that naturally group these subjects. In other words, there may not be any model that well separates the people with the two conditions.     

Given the foregoing, notice the following:
1. As is very often the case, there is no 'knee' in the WCSS curve, just a slow change in slope.     
2. There are several maximums of the silhouette coefficient all within a narrow range of values, at 2, 4 and 6.
3. The Davies-Bouldin index has minimum at 6 and 8. The slightly lower values at higher $k$s likely arise from fragmentation of the clusters, given the small sample size.
4. The Calinski Harabasz index monotonically decreases, with a hardly noticeable break at $k=4$. The monotonic decrease of this metric likely indicating a tendency toward cluster fragmentation, given the small sample size.
   
Considering all of the above, we can select $k=6$ or $k=4$. another possible choice. Faced with the conflicting metrics, we will chose the lower number of clusters to prevent over-fitting and cluster fragmentation.   

To test and evaluate the 4-cluster model, execute the code in the cell below. 

In [ ]:
gene_projection = pd.DataFrame({'component1':gene_pca_transform['0'],
                               'component2':gene_pca_transform['1'],
                               'disease':labels}) 

n_clusters=[4,6]
for n_cluster in n_clusters:   
    if 'labels' in gene_pca_transform.columns: gene_pca_transform.drop('labels', axis=1, inplace=True)
    gene_projection['cluster_assignments'] = KMeans(n_clusters=n_cluster, n_init=10).fit_predict(gene_pca_transform)

    plot_cluster_assignments(gene_projection,  
                         style = 'disease',
                         s=50, 
                         alpha=0.7, 
                         title = 'PCA projection of gene response cluster assignments, k = ' + str(n_cluster))
    plt.show()

    print(gene_projection.loc[:,['cluster_assignments','disease']].value_counts().sort_index())

The $k=6$ model shows some interesting relationships. Crohn's is the minority case, coded as $0$, making this problem harder yet. All the clusters with most of the Crohn's cases include some colitis cases as well. There is one cluster with only colitis cases. Equally important as the separation of the case within clusters is the compactness and separation of the clusters themselves. One of these clusters with only colitis cases, is fairly compact and shows good separation with the others While three of the clusters from a band showing significant overlap. Overall these clusters show reasonable compactness and no fragmentation.   

The $k=6$ model does not separate the two diseases better than the $k=4$ model.  

A next step is to try a $k=6$ model. We leave this to those interested.   

## UMAP projection and clustering of gene data

The UMAP algorithm allows us to **map from a non-Euclidean space to a Euclidean space**. Here we test **mapping from cosine distance to Euclidean distance**. You can now examine the result of changing the hyperparameter values for the embedding of the gene data. As a first step, we execute the code in the cell below to display a baseline projection using cosine distance and the default hyperparameters for `n_neighbors` and `min_dist`. 

In [ ]:
def plot_pca(X, labels, ax=None):
    pca_projected = pd.DataFrame(X, columns=['Component_1','Component_2'])
    pca_projected['labels'] = labels 
    if ax == None: 
        sns.scatterplot(data=pca_projected, x='Component_1', y='Component_2', hue='labels')
    else:       
        sns.scatterplot(data=pca_projected, x='Component_1', y='Component_2', hue='labels', ax=ax)
    return pca_projected

np.random.seed(8833)
gene_UMAP = umap.UMAP(metric='cosine').fit_transform(gene_data)
umap_projected=plot_pca(gene_UMAP, labels)
plt.show()

Next, execute the code in the cell below to display the embeddings for the pairs of values for `n_neighbors` and `min_dist`.

In [ ]:
min_dist_list = [0.5, 0.5, 0.02, 0.02]
n_neighbor_list = [10,50,10,50]
_, ax = plt.subplots(2,2, figsize=(10,10))
ax = ax.flatten()
np.random.seed(1010)
for i, (min_dist, n_neighbors) in enumerate(zip(min_dist_list,n_neighbor_list)):
    gene_UMAP = umap.UMAP(min_dist=min_dist, n_neighbors=n_neighbors, metric='cosine').fit_transform(gene_data)
    umap_projected=plot_pca(gene_UMAP, labels, ax=ax[i])
    ax[i].set_title('min_dist = ' + str(min_dist) + '  n_neihbors = ' + str(n_neighbors))
plt.show()

Examine these plots and notice the following:      
1. The small value of `n_neighors` leads to tighter groups. Whereas, the large values of n_neighbors leads to large, poorly separated groups. This represents the fact that the the `n_neighbors` hyperparameter affects how local or wide the search for nearest neighbors is, which affects the properties of the graph used for the UMAP algorithm.      
2. The smaller value of `min_dist` yields tighter groups. This is expected since the distance between samples on the manifold is smaller.     

### UMAP dimensionality reduction for non-Euclidean agglomerative clustering

We will now create a reduced dimensionality manifold projection with UMAP and apply agglomerative clustering in this Euclidean space. A 20 dimensional space is used for the clustering, which is a reasonable dimensionality for the k-means algorithm.  The code in the cell below does the following:   
1. Compute a 40 component embedding using cosine distance with `n_neighbors=10` and `min_dist=0.05`.
2. Create a data frame from the projection.
3. Call the cluster search function is used over a range of $k=2$ to $k=20$.

Execute the code in the cell below.     

In [ ]:
n_components = 40  
n_neighbors= 10
min_dist = 0.05
np.random.seed(5647)
umap_transform = umap.UMAP(min_dist=min_dist, n_neighbors=n_neighbors, n_components=n_components, metric='cosine')
gene_umap_transform = umap_transform.fit_transform(gene_data)
gene_umap_transform = pd.DataFrame(gene_umap_transform, columns=[str(i) for i in range(n_components)])
    
np.random.seed(8686)
evaluate_agglomerative_clusters(gene_umap_transform, nclusts=(2,20)) 
plt.show()

As is usually the case, finding a value of k in these charts is ambiguous. We can observe the following.     
1. Silhouette index has peaks at 2, 4 and 8 clusters.
2. Maximum cluster diameter monotonically decreases and but shows a clear breaks in slope at 4 and 8 clusters.     
Given the above, we test models with 4 and 8 clusters.  

The code in the cell below computes an agglomerative clustering model for 4 and 8 clusters, using the 40 dimension UMAP manifold embedding, and displays the results. Execute this code.  

In [ ]:
np.random.seed(465)
umap_embedding = umap.UMAP(min_dist=min_dist, n_neighbors=n_neighbors, n_components=2, metric='cosine')
gene_embedding_df = umap_embedding.fit_transform(gene_data)

gene_embedding_df = pd.DataFrame(gene_embedding_df, columns = ['component1', 'component2'])
gene_embedding_df['disease'] = [0 if label=='Ulcerative Colitis (UC)' else 1 for label in labels]  


n_clusters=[4, 8]
for n_cluster in n_clusters:   
    if 'cluster_assignments' in gene_umap_transform.columns: gene_umap_transform.drop(columns='cluster_assignments', inplace=True)
    model_agglomerative =  AgglomerativeClustering(n_clusters=n_cluster, linkage='complete', 
                                               metric='euclidean', compute_full_tree=False)   
    np.random.seed(298)
    %time gene_embedding_df['cluster_assignments'] = model_agglomerative.fit_predict(gene_umap_transform)

    plot_cluster_assignments(gene_embedding_df,  
                         style = 'disease',
                         s=50, 
                         alpha=0.7, 
                         title = 'UMAP projection of gene response cluster assignments, n_cluster = ' + str(n_cluster))
    plt.show()

    print(gene_embedding_df.loc[:,['cluster_assignments','disease']].value_counts().sort_index())


> **Exercise 3-9:** Examine these results of this model and answer these questions:
> 1. Why is the Euclidean distance metric used for the agglomerative clustering algorithm, when we are using cosine distance for the overall model, including the UMAP dimensionality reduction.    
> 2. Does the agglomerative clustering with with 8 clusters show a tendency toward cluster fragmentation, when compared to the model with 4 clusters, and based on what evidence?
> 3. How do the cases of Crohn's vs. colitis separate with the agglomerative clustering with cosine distance compare to the Euclidean k-means clusters and what does this tell you about how well the genetic responses separate these cases?

> **Answers:**
> 1.      
> 2.   
> 3.    

#### Copyright 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026 Stephen F Elston. All rights reserved. 